## Evolution of $w_{g,j}(p^\#)$
Here we calculate and graph the evolution of $w_{g,j}(\lambda)$ for $p_0=37$ and gaps $2 \le g \le 82$.
The evolution is shown as a heatmap.  The asymptotic values are shown at the left.

This work is related to calculating $\delta_s(\lambda)$ in our studies of quadratic density.

In [1]:
import numpy as np
from numpy.polynomial.polynomial import polyval
import array
import matplotlib as mpl

import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import pickle

import gc
import psutil
import sys

import itertools
from ipywidgets import interact
import ipywidgets as widgets
from IPython.display import display
plt.ion


<function matplotlib.pyplot.ion() -> 'AbstractContextManager'>

In [2]:
# load the array of primes and of the corresponding values of lambda
smallprimes = np.load('primesE9.npy')
lambdaE9 = np.load('lambdaE9.npy')

In [3]:
# p0=37 for the models, so lambdaE9[j] corresponds to smallprimes[j+10]
print(f"{len(smallprimes)} primes from {smallprimes[0]} to {smallprimes[-1]}")
print(f"{len(lambdaE9)} values of lambda from {lambdaE9[0]} to {lambdaE9[-1]}")
print(f"smallprimes[10] is {smallprimes[10]}")

51961884 primes from 3 to 1023101273
51961874 values of lambda from 1.0 to 0.17990704950441305
smallprimes[10] is 37


In [4]:
lambdaE9[0:100]

array([1.        , 0.97435897, 0.95059412, 0.92946981, 0.91124491,
       0.89525816, 0.88008429, 0.86654453, 0.85398591, 0.84195794,
       0.83102343, 0.82076388, 0.81132981, 0.8027895 , 0.79468051,
       0.78681239, 0.77931894, 0.77203558, 0.76508031, 0.75895966,
       0.75307625, 0.74749791, 0.74204172, 0.73699382, 0.73204755,
       0.72732467, 0.72280712, 0.71842647, 0.71422515, 0.71018998,
       0.70622244, 0.70248581, 0.69880787, 0.69522424, 0.69169519,
       0.68838564, 0.68527077, 0.68222512, 0.67921973, 0.67627938,
       0.67342588, 0.6706082 , 0.667915  , 0.66529572, 0.66274669,
       0.6602645 , 0.65780998, 0.65541795, 0.65306878, 0.65074469,
       0.64850845, 0.6463822 , 0.64429034, 0.64221867, 0.64017988,
       0.63823405, 0.63632887, 0.63448444, 0.63265595, 0.63085352,
       0.62908642, 0.62736289, 0.62567189, 0.62401228, 0.62237445,
       0.62076625, 0.61919469, 0.61764282, 0.61612528, 0.61464776,
       0.61318082, 0.61175149, 0.61033211, 0.60893547, 0.60755

In [5]:
# Merten's Third Theorem:  lambda(p) = C / ln(p)  or  p = exp(C/lambda)
# for large primes p
MertensC = lambdaE9[-1] * np.log(smallprimes[-1])
MertensC

np.float64(3.7323704160415)

In [6]:
# The matrix of coefficients lj is created in the '10_wg' notebook.  Here we read the result from file.
# These coefficients include the alternating signs
ljmat = np.load('lj37.npy')
ljmat = np.abs(ljmat)
ljmat.shape

(19, 41)

In [7]:
# The coefficient matrix uses p0=37=smallprimes[10].
# ljmat[i,j] is the (i+1)st coefficient for gap g=2(j+1)
ljmat[:,40]

array([1.02564100e+00, 1.15539420e+01, 6.04104830e+01, 1.94488465e+02,
       4.31258857e+02, 6.97937481e+02, 8.52214506e+02, 8.00382107e+02,
       5.84027075e+02, 3.32122850e+02, 1.46758457e+02, 4.99397940e+01,
       1.28839350e+01, 2.46138300e+00, 3.36767000e-01, 3.15410000e-02,
       1.91000000e-03, 7.00000000e-05, 1.00000000e-06])

In [9]:
# 19x19 matrix of right eigenvectors -- an upper triangular Pascal matrix
# gaps up to g=82 have admissible driving terms of length at most 19
eigR=np.array([[1, -1, 1, -1, 1, -1, 1, -1, 1,  -1,   1,  -1,   1,   -1,    1,  -1,      1,    -1,     1],
                [0, 1,-2, 3, -4, 5, -6, 7, -8,   9, -10,  11, -12,   13,  -14,  15,    -16,    17,   -18],
                [0, 0, 1,-3, 6,-10,15,-21, 28, -36,  45, -55,  66,  -78,   91, -105,   120,  -136,   153],
                [0, 0, 0, 1, -4,10,-20,35,-56,  84,-120, 165, -220, 286, -364,  455,  -560,   680,  -816],
                [0, 0, 0, 0, 1, -5,15,-35, 70,-126, 210,-330, 495, -715, 1001,-1365,  1820, -2380,  3060],
                [0, 0, 0, 0, 0, 1, -6, 21, -56, 126,-252, 462,-792, 1287,-2002, 3003, -4368,  6188, -8568],
                [0, 0, 0, 0, 0, 0, 1,  -7, 28, -84, 210,-462, 924,-1716, 3003,-5005,  8008,-12376, 18564],
                [0, 0, 0, 0, 0, 0, 0,   1, -8,  36,-120, 330,-792, 1716,-3432, 6435,-11440, 19448,-31824],
                [0, 0, 0, 0, 0, 0, 0, 0,    1,  -9,  45,-165, 495,-1287, 3003,-6435, 12870,-24310, 43758],
                [0, 0, 0, 0, 0, 0, 0, 0, 0,      1, -10,  55,-220,  715,-2002, 5005,-11440, 24310,-48620],
                [0, 0, 0, 0, 0, 0, 0, 0, 0,  0,       1, -11,  66, -286, 1001,-3003,  8008,-19448, 43758],
                [0, 0, 0, 0, 0, 0, 0, 0, 0,  0,  0,        1, -12,   78, -364, 1365, -4368, 12376,-31824],
                [0, 0, 0, 0, 0, 0, 0, 0, 0,  0,  0,  0,         1,  -13,   91, -455,  1820, -6188, 18564],
                [0, 0, 0, 0, 0, 0, 0, 0, 0,  0,  0,  0,  0,           1,  -14,  105,  -560,  2380, -8568],
                [0, 0, 0, 0, 0, 0, 0, 0, 0,  0,  0,  0,  0,   0,            1,  -15,   120,  -680,  3060],
                [0, 0, 0, 0, 0, 0, 0, 0, 0,  0,  0,  0,  0,   0,   0,             1,   -16,   136,  -816],
                [0, 0, 0, 0, 0, 0, 0, 0, 0,  0,  0,  0,  0,   0,   0,   0,               1,   -17,   153],
                [0, 0, 0, 0, 0, 0, 0, 0, 0,  0,  0,  0,  0,   0,   0,   0,    0,                1,   -18],
                [0, 0, 0, 0, 0, 0, 0, 0, 0,  0,  0,  0,  0,   0,   0,   0,    0,    0,                 1]])

In [10]:
# block to check the available system memory
gc.collect()
memory = psutil.virtual_memory()
available_memory = memory.available
del memory
print(f"Available memory: {available_memory / (1024 ** 2):.2f} MB")

Available memory: 3484.30 MB


In [11]:
dex= 100
smallprimes[dex+10],lambdaE9[(dex-5):(dex+5)]

(np.int64(613),
 array([0.58300397, 0.5820175 , 0.58104259, 0.58007257, 0.57911378,
        0.57816596, 0.57722585, 0.57629032, 0.57537412, 0.57447369]))

In [12]:
# global variables - set up an array of p(lambdaE9) values
lamarray = np.zeros(100)
plamarray = np.zeros(100)

lamlen = len(lambdaE9)

lamswitch = lambdaE9[-1]  # the last value we have calculated directly, then we switch to Mertens' Third Theorem

# Array lamarray:  20 values to 0.05, 30 values to 0.18, 30 values to 0.75, 20 values to 1.0
# the top of the lamarray is the start of the smallprimes array
i = 99
j = 0
while i >= 80:
    lamarray[i] = lambdaE9[j]
    plamarray[i] = smallprimes[j+10]
    # print(f"i {i:2} val {lamarray[i]:.3f} p {plamarray[i]:6} j {j:5} ")
    i -= 1
    j += 1

# for the next section copy appropriate values from smallprimes array
lamstep = round((lamarray[i+1]-0.18)/30,3)
curlam = round(lamarray[i+1],3)-lamstep
while i >= 50:
    while (j< lamlen) and (lambdaE9[j] > curlam) and (curlam >= 0.175):
        j += 1
    lamarray[i] = lambdaE9[j]
    plamarray[i] = smallprimes[j+10]
    # print(f"i {i:2} val {lamarray[i]:.3f} p {plamarray[i]:8.0f} j {j:5}")
    i -= 1
    curlam -= lamstep

# shorten the step and transition to Mertens' Third Theorem
lamstep = round((0.13/30),4)
curlam = round(lamarray[i+1]-lamstep,4)
while i > 20:
    lamarray[i] = curlam
    plamarray[i] = np.exp(MertensC / curlam)
    # print(f"i {i:2}  val {lamarray[i]:.3f} p {plamarray[i]:10.4e}")
    i -= 1
    curlam -= lamstep

lamstep = round(((lamarray[i+1]-0.0035)/(i+2)),4)
curlam = round(lamarray[i+1]-lamstep,4)
while (i >= 0) and (curlam > 0) and (i >=0):
    lamarray[i] = curlam
    plamarray[i] = np.exp(MertensC / curlam)
    # print(f"i {i:2} val {lamarray[i]:.3f} p {plamarray[i]:10.4e}")
    i -= 1
    curlam -= lamstep

i=0
while i < 100:
    lamarray[i] = round(lamarray[i],4)
    i += 1

i = 0
while i < 100:
    if plamarray[i] < 1000000000:
        print(f"i {i:2} val {lamarray[i]:.4f} p {plamarray[i]:10.0f}")
    else:
        print(f"i {i:2} val {lamarray[i]:.4f} p {plamarray[i]:10.3e}")
    i += 1

i  0 val 0.0055 p 5.222e+294
i  1 val 0.0083 p 1.972e+195
i  2 val 0.0111 p 1.075e+146
i  3 val 0.0139 p 4.121e+116
i  4 val 0.0167 p  1.155e+97
i  5 val 0.0195 p  1.335e+83
i  6 val 0.0223 p  4.878e+72
i  7 val 0.0251 p  3.798e+64
i  8 val 0.0279 p  1.255e+58
i  9 val 0.0307 p  6.304e+52
i 10 val 0.0335 p  2.435e+48
i 11 val 0.0363 p  4.510e+44
i 12 val 0.0391 p  2.861e+41
i 13 val 0.0419 p  4.854e+38
i 14 val 0.0447 p  1.832e+36
i 15 val 0.0475 p  1.334e+34
i 16 val 0.0503 p  1.681e+32
i 17 val 0.0531 p  3.360e+30
i 18 val 0.0559 p  9.938e+28
i 19 val 0.0587 p  4.112e+27
i 20 val 0.0615 p  2.274e+26
i 21 val 0.0643 p  1.619e+25
i 22 val 0.0686 p  4.256e+23
i 23 val 0.0729 p  1.719e+22
i 24 val 0.0772 p  9.925e+20
i 25 val 0.0815 p  7.743e+19
i 26 val 0.0858 p  7.801e+18
i 27 val 0.0901 p  9.785e+17
i 28 val 0.0944 p  1.483e+17
i 29 val 0.0987 p  2.648e+16
i 30 val 0.1030 p  5.462e+15
i 31 val 0.1073 p  1.278e+15
i 32 val 0.1116 p  3.347e+14
i 33 val 0.1159 p  9.677e+13
i 34 val 0.120

In [13]:
# Set the color map based on asymptotic values
#  For maxval=8/3  then val=1 @ 0.375 (red), val=4/3 @ 0.5 (green), val=2 @ 0.75 (blue), val=8/3 @ 1.0 (gold)
cdict = {
    'red': (
        (0.0, 1.0, 1.0),
        (0.0000001, 1.0, 1.0),
        (0.000001, 0.4, 0.4),
        (0.37, 1.0, 1.0),  # if maxval=8/3, then 0.375 corresponds to val=1
        (0.38, 1.0, 1.0),
        (0.39, 0.0, 0.0),
        (0.75, 0.0, 0.0),
        (0.755, 0.4, 0.4),
        (0.98, 1.0, 1.0),
        (1.0, 1.0, 1.0)),
    'green': (
        (0.0, 1.0, 1.0),
        (0.000001, 1.0, 1.0),
        (0.01, 0.0, 0.0),
        (0.37, 0.0, 0.0),
        (0.38, 0.4, 0.4),
        (0.49, 1.0, 1.0),
        (0.51, 1.0, 1.0),
        (0.65, 0.0, 0.0),
        (0.75, 0.0, 0.0),
        (0.755, 0.2, 0.2),
        (0.98, 0.84, 0.84),
        (1.0, 0.84, 0.84)),
    'blue': (
        (0.0, 1.0, 1.0),
        (0.000001, 1.0, 1.0),
        (0.01, 0.0, 0.0),
        (0.5, 0.0, 0.0),
        (0.505, 0.4, 0.4),
        (0.74, 1.0, 1.0),
        (0.76, 1.0, 1.0),
        (0.8, 0.0, 0.0),
        (1.0, 0.0, 0.0))}

mpl.colormaps.register(LinearSegmentedColormap('gapcolors', cdict))
        

In [14]:
# Set the color map for delta_(g,j) based on asymptotic values
#  For maxval=20  then val=1 @ 0.05 (red), val=2 @ 0.1 (green), val=10 @ 0.5 (blue), val=20 @ 1.0 (gold)
cdict = {
    'red': (
        (0.0, 1.0, 1.0),
        (0.0000001, 1.0, 1.0),
        (0.000001, 0.4, 0.4),
        (0.045, 1.0, 1.0),  # if maxval=8/3, then 0.375 corresponds to val=1
        (0.055, 1.0, 1.0),
        (0.058, 0.0, 0.0),
        (0.5, 0.0, 0.0),
        (0.55, 0.4, 0.4),
        (0.98, 1.0, 1.0),
        (1.0, 1.0, 1.0)),
    'green': (
        (0.0, 1.0, 1.0),
        (0.000001, 1.0, 1.0),
        (0.002, 0.0, 0.0),
        (0.05, 0.0, 0.0),
        (0.055, 0.4, 0.4),
        (0.1, 1.0, 1.0),
        (0.13, 1.0, 1.0),
        (0.15, 0.0, 0.0),
        (0.5, 0.0, 0.0),
        (0.55, 0.2, 0.2),
        (0.98, 0.84, 0.84),
        (1.0, 0.84, 0.84)),
    'blue': (
        (0.0, 1.0, 1.0),
        (0.000001, 1.0, 1.0),
        (0.002, 0.0, 0.0),
        (0.1, 0.0, 0.0),
        (0.105, 0.4, 0.4),
        (0.48, 1.0, 1.0),
        (0.52, 1.0, 1.0),
        (0.55, 0.0, 0.0),
        (1.0, 0.0, 0.0))}

mpl.colormaps.register(LinearSegmentedColormap('delcolors', cdict))
        

In [15]:
# global variables describing the data for the figures
# wgj is the 2D array of relative populations.  Note that the indices are the transpose to the array ljmat[]
# wgj[i,0] is the asymptotic population for g=2(i+1).
wgj = np.zeros((41,20))

delgj = np.zeros((41,20))

i=0
while i < 41:
    wgj[i,0] = ljmat[0,i]
    i += 1

gapticks = np.arange(0,41)
gapnames = np.arange(2,84, 2, dtype=int)
gaplabels = gapnames[:].astype(str)

lenticks = np.arange(0,20)
lenlabels = lenticks[:].astype(str)
lenlabels[0] = 'Inf'

wgjlength = np.zeros(41, dtype=int)
i = 0
while i < 41:
    j=0
    while (j < 19) and (np.abs(ljmat[j,i]) > 0.0):
        j += 1
    # print(f"gap {2*i+2} {i} len {j} abs {np.abs(ljmat[(j-1),i])} {np.abs(ljmat[(j-1),i]) > 0.0}", end='\r') 
    wgjlength[i] = j
    i += 1
    

In [16]:
wgjlength[:]

array([ 1,  1,  2,  3,  3,  4,  4,  5,  5,  6,  5,  6,  7,  7,  8,  9,  9,
       10,  9, 10, 11, 11, 11, 12, 13, 12, 13, 14, 14, 15, 15, 15, 16, 16,
       17, 17, 17, 18, 18, 19, 19])

In [17]:

def draw_wgj(lamval, showdelta):   # the input parameter is lambda(p)
    global wgj
    global ljmat
    global lambdaE9
    global smallprimes
    global lamswitch
    global gaplabels
    global wgjlength

    
    plt.ioff()  # turn interactive mode off
 
    
    fig, ax = plt.subplots()
    fig.set_size_inches(12,8)

    # fix lamval if lamval is in the calculated range
    i=0
    while (i < 100) and (lamval > lamarray[i]):
        i += 1
    ptitle = plamarray[i]

    # calculate the table of relative populations
    coeffs = np.zeros(19) # array for coefficients lj*lambda^(j-1)
    ig = 0  # index for gap
    while ig < 41:
        j = 0  # index within array
        lamk = 1.0
        while j < 19:
            coeffs[j] = ljmat[j,ig]*lamk
            lamk *= lamval
            j += 1
        wgrow = np.dot(eigR, coeffs)
        wgj[ig,1:] = wgrow
        ig += 1

    if showdelta:
        ig=0
        while ig < 41:
            delgj[ig,0]=0
            j=1
            while j < 19:
                if (wgj[ig,j] > 0):
                    delgj[ig,j] = wgj[ig,j+1]/wgj[ig,j]
                else:
                    delgj[ig,j] = 0
                j += 1
            ig += 1
        imwg = plt.imshow(delgj, cmap='delcolors', vmax=20, vmin=0, aspect=0.4)
    else:
        imwg = plt.imshow(wgj, cmap='gapcolors', vmax=2.667, vmin=0, aspect=0.4)  # 
            
    plt.xticks(lenticks, labels=lenlabels)
    plt.yticks(gapticks,labels=gaplabels)
    fig.colorbar(imwg, ax=ax)
    
    if ptitle > 1000000000:
        ax.set_title(f"Relative populations of driving terms for gaps 2 to 82 at lambda={lamval:.4f} or primes around {ptitle:.3e}")
    else:
        ax.set_title(f"Relative populations of driving terms for gaps 2 to 82 at lambda={lamval:.4f} or primes around {ptitle:6.0f}")

    plt.show()
    # for debugging & tuning : print(wgj)

# Interactive controls
xlamvalSelect = widgets.SelectionSlider(value=lamarray[98], options=lamarray, 
                  description="lambda_par", layout=widgets.Layout(width='80%'), disabled=False)

xshowdelta = widgets.Checkbox(value=False, description='Show delta', disabled=False)

# save figure checkbox... XXXQHERE

interact(draw_wgj, lamval=xlamvalSelect, showdelta=xshowdelta)


interactive(children=(SelectionSlider(description='lambda_par', index=98, layout=Layout(width='80%'), options=…

<function __main__.draw_wgj(lamval, showdelta)>